# streaming_ingest-gh_archive-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
schema = StructType().add("id", StringType()).add("type", StringType()).add("created_at", StringType())
stream = spark.readStream.schema(schema).json("s3a://landing/gh_archive")

## 4. Transform

In [ ]:
events = stream.withColumn("created_at", F.col("created_at").cast("timestamp"))

## 5. Write

In [ ]:
query = events.writeStream.format("iceberg").outputMode("append").option("checkpointLocation", "s3a://checkpoints/gh_events_file").toTable("lakehouse.bronze.gh_events_stream")
# query.awaitTermination() to keep stream running

## 6. Verify

In [ ]:
spark.table("lakehouse.bronze.gh_events_stream").count()